# 05. Relatório Final

**Input:** `metrics.json`, `data/quality_report.md`, gráficos gerados nos notebooks anteriores  
**Output:** resumo estruturado no formato do relatório Vale

Seções conforme template:
1. Resumo
2. Introdução
3. Entendimento do Negócio
4. Metodologia
5. Resultados e Discussões
6. Conclusão e Trabalhos Futuros

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

DATA_DIR = Path('../data')
SHAP_DIR = Path('../shap_plots')

## 1. Resumo

Este trabalho desenvolveu uma solução de análise preditiva capaz de antecipar alertas críticos 
(**Don't Go**) em equipamentos de mineração (caminhões e escavadeiras) da Vale S.A.

A partir de 37 milhões de registros de telemetria (Jan-Jun 2025), foram identificados **3 erros 
intencionais de qualidade de dados**, construído um pipeline ETL reproduzível e treinado um modelo 
XGBoost capaz de prever ocorrências de Don't Go nos próximos 1h, 2h e 4h.

A métrica principal é **F1-score** (não accuracy), dado o extremo desbalanceamento de classes (~0,05% positivo).

## 2. Diagnóstico de Qualidade de Dados

Foram identificados e corrigidos os seguintes erros intencionais:

In [ ]:
report_path = DATA_DIR / 'quality_report.md'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
else:
    print('Execute o notebook 01_clean.ipynb primeiro para gerar o quality_report.md.')

### Tabela de Erros

| # | Coluna | Problema | Registros Afetados | Correção |
|---|--------|----------|-------------------|----------|
| 1 | `Criticidade` | Corrupção UTF-8: "N??o Crítico" e "Não Cr??tico" | ver report | Regex normaliza para "Não Crítico" |
| 2 | `Classe` | String literal `"NULL"` em vez de `null` real | ~5,3M (Jan) | `replace('NULL', np.nan)` |
| 3 | `Valor` | Vírgula como separador decimal | ver report | `str.replace(',', '.')` + cast float64 |

## 3. Entendimento do Negócio

In [ ]:
imgs = ['eda_criticidade_tipo.png', 'eda_dontgo_timeline.png', 'eda_top_alarmes.png']
for img_name in imgs:
    img_path = DATA_DIR / img_name
    if img_path.exists():
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.imshow(mpimg.imread(str(img_path)))
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{img_name} não encontrado. Execute notebook 02_eda.ipynb.')

## 4. Metodologia

### Pipeline

```
Raw Parquets → load_telemetry() → clean_telemetry() → build_features() → train_model()
                                  ↓
                           QualityReport
```

### Decisões Técnicas

| Decisão | Escolha | Racional |
|---------|---------|----------|
| Modelo | XGBoost | Gradient boosting supera redes neurais em dados tabulares |
| Split | Temporal (Jan-Abr / Mai-Jun) | Série temporal, random split vazaria informação do futuro |
| Desbalanceamento | `scale_pos_weight` | Nativo no XGBoost, sem overhead de SMOTE |
| Métrica | F1 + Precision + Recall | Accuracy é enganosa com 0,05% positivo |
| Horizontes | 1h, 2h, 4h | Explorado empiricamente; melhor horizonte reportado |

## 5. Resultados

In [ ]:
metrics_path = Path('../metrics.json')
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    best = metrics['best_label']
    print(f'Melhor horizonte: {best}')
    print('\nMétricas por horizonte:')
    for horizon, m in metrics['all_horizons'].items():
        print(f'  {horizon}: F1={m["f1"]:.3f}  Precision={m["precision"]:.3f}  Recall={m["recall"]:.3f}  AUC={m["auc"]:.3f}')
    print('\nAvaliação por threshold (melhor horizonte):')
    for r in metrics.get('thresholds', []):
        print(f'  threshold={r["threshold"]}  F1={r["f1"]:.3f}  Precision={r["precision"]:.3f}  Recall={r["recall"]:.3f}')
else:
    print('Execute o notebook 04_model.ipynb primeiro.')

In [ ]:
# SHAP plots
for img_name in SHAP_DIR.glob('*.png'):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(mpimg.imread(str(img_name)))
    ax.axis('off')
    ax.set_title(img_name.stem)
    plt.tight_layout()
    plt.show()

## 6. Conclusão e Trabalhos Futuros

### Conclusão

O pipeline desenvolvido demonstra que é possível antecipar ocorrências de Don't Go com 
antecedência de até 4 horas, utilizando exclusivamente features derivadas do histórico 
de alarmes de telemetria. O modelo XGBoost com split temporal respeita as restrições de 
série temporal e oferece explicabilidade via SHAP.

### Trabalhos Futuros

- **LightGBM como alternativa:** comparação direta com XGBoost nos 3 horizontes
- **SMOTE como ablação:** testar oversampling sintético vs. `scale_pos_weight`
- **Features adicionais:** distribuição de alarmes por turno, frequência de manutenções recentes
- **Dashboard Streamlit:** interface para operadores consultarem risco por TAG em tempo real
- **Detecção de concept drift:** monitorar distribuição das features ao longo do tempo